In [40]:
import pandas as pd 
import numpy as np 
from scipy import stats
import matplotlib.pyplot as plt 

### Table 8.8

In [41]:
df_T808 = pd.DataFrame({
    'replicate': np.repeat(np.arange(2)+1, 16),
    'run': np.repeat(np.arange(8) + 1, 4),
    'position': np.tile(np.arange(4) + 1, 8),
    'treatment': [i for i in 'ABCDCDABDCBABADCABCDCDABDCBABADC'],
    'holder': [int(i) for i in '12344321214334121234432121433412'],
    'paper': [i for i in 'abcdbadccdabdcbaefghfehgghefhgfe'],
    'wearing': [320,297,299,313,
                266,227,260,240,
                221,240,267,252,
                301,238,243,290,
                285,280,331,311,
                268,233,291,280,
                265,273,234,243,
                306,271,270,272]
})

In [42]:
df_T808['grand_average'] = df_T808['wearing'].mean()

In [43]:
df_T808_col_transf = df_T808.columns[:6]

In [44]:
for i in df_T808_col_transf:
    df_T808[f'{i}_deviation'] = df_T808.groupby(i)['wearing'].transform('mean') - df_T808['grand_average']

In [45]:
# adjust the run and paper deviations due to 2 replicates introduced new runs and new papers
df_T808['run_deviation'] = df_T808['run_deviation'] - df_T808['replicate_deviation']
df_T808['paper_deviation'] = df_T808['paper_deviation'] - df_T808['replicate_deviation']

In [46]:
df_T808['residual'] = df_T808['wearing'] - df_T808[['grand_average']+[i for i in df_T808.columns if 'deviation' in i]].sum(axis=1)

In [47]:
df_T808_ANOVA = (df_T808[df_T808.columns[-7:]]**2).sum().to_frame(name='sum_square').reset_index(names='variance_source')

In [48]:
df_T808_ANOVA['degree_of_freedom'] = [1,6,3,3,3,6,9]

In [49]:
df_T808_ANOVA['mean_square'] = df_T808_ANOVA['sum_square'] / df_T808_ANOVA['degree_of_freedom']

In [50]:
df_T808_ANOVA['ratio_mean_square'] = df_T808_ANOVA['mean_square'] / df_T808_ANOVA.loc[6,'mean_square']

In [51]:
df_T808_ANOVA['sf'] = df_T808_ANOVA.apply(lambda row: stats.f.sf(row['ratio_mean_square'], dfn=row['degree_of_freedom'], dfd=df_T808_ANOVA.loc[6,'degree_of_freedom']), axis=1)

In [52]:
df_T808_ANOVA.sum_square.sum() + 32*df_T808.loc[0,'grand_average']**2

np.float64(2384713.0)

In [53]:
(df_T808.wearing**2).sum()

np.int64(2384713)

In [54]:
df_T808_ANOVA

,variance_source,sum_square,degree_of_freedom,mean_square,ratio_mean_square,sf
0,replicate_deviation,603.78125,1,603.781250,5.725872,0.040366
1,run_deviation,14770.43750,6,2461.739583,23.345550,0.000053
2,position_deviation,2217.34375,3,739.114583,7.009286,0.009925
3,treatment_deviation,1705.34375,3,568.447917,5.390793,0.021245
4,holder_deviation,109.09375,3,36.364583,0.344858,0.793790
5,paper_deviation,6108.93750,6,1018.156250,9.655537,0.001698
6,residual,949.03125,9,105.447917,1.000000,0.500000


### Build Table 8.8

In [55]:
def get_square(seq: list, n, m, i=1):
    '''seq: sequence with the length equal m, the number of cols. 
    n is the number of rows. i is the number to shift the initial seq btw rows (default i=1)'''
    seq_arr = np.array([(np.arange(m) + i) % n for i in range(n)])
    row_order, col_order = np.arange(n), np.arange(m)
    np.random.shuffle(row_order)
    np.random.shuffle(col_order)
    seq_arr = seq_arr[:,col_order][row_order,:]
    return np.array(seq)[seq_arr]


In [56]:
row_T808, col_T808 = np.indices((8,4))
replicate_T808 = ['I'] * 16 + ['II'] * 16
treatment_T808 = get_square([i for i in 'ABCD'], 4, 4)
holder_T808 = get_square([1,2,3,4], 4, 4)
paper_T808_1 = get_square([i for i in 'abcd'],4, 4)
paper_T808_2 = get_square([i for i in 'efgh'],4, 4)

In [57]:
bdf_T808 = pd.DataFrame({
    'row': row_T808.flatten(),
    'col': col_T808.flatten(),
    'replicate': replicate_T808,
    'treatment': np.tile(treatment_T808.flatten(), 2),
    'holder': np.tile(holder_T808.flatten(), 2),
    'paper': np.concat((paper_T808_1.flatten(), paper_T808_2.flatten())),
})

In [ ]:
bdf_T808

,row,col,replicate,treatment,holder,paper
0,0,0,I,D,4,b
1,0,1,I,C,2,c
2,0,2,I,B,3,a
3,0,3,I,A,1,d
4,1,0,I,A,1,a
5,1,1,I,D,3,b
6,1,2,I,C,4,d
7,1,3,I,B,2,c
8,2,0,I,C,2,d
9,2,1,I,B,4,a


: 